# 05 — Model Optimization & Advanced Evaluation

**AI Smart Inventory Management & Demand Forecasting**  
**Phase 5 — Day 9 (Missing Tasks)**  
**Sprint 2**

---

## Objectives

1. Train additional ML models: Linear Regression, Gradient Boosting, (optional) XGBoost  
2. Perform RandomizedSearchCV hyperparameter tuning on Random Forest  
3. Evaluate all models on MAE, MSE, RMSE, R², MAPE  
4. Generate Feature Importance analysis and charts  
5. Produce model comparison visualizations  
6. Serialize the champion model and write metadata

---

## 0. Setup & Imports

In [ ]:
import os
import sys
import json
import warnings
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import joblib

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

# Path setup
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..', '..'))
DATA_PROCESSED = os.path.join(PROJECT_ROOT, 'data', 'processed')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models')
FIGURES_DIR = os.path.join(PROJECT_ROOT, 'reports', 'figures')

for d in [MODELS_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Data path    : {DATA_PROCESSED}')
print(f'Models dir   : {MODELS_DIR}')
print(f'Figures dir  : {FIGURES_DIR}')

## 1. Feature Configuration (matching train_forecast.py)

In [ ]:
VALIDATION_DAYS = 60
LAG_FEATURES = [1, 7, 14, 30]
ROLLING_WINDOWS = [7, 14, 30]
RANDOM_STATE = 42

FEATURE_COLS = (
    [f'lag_{d}' for d in LAG_FEATURES]
    + [f'rolling_mean_{w}' for w in ROLLING_WINDOWS]
    + [f'rolling_std_{w}' for w in ROLLING_WINDOWS]
    + ['day_of_week', 'month', 'quarter', 'is_weekend']
)

print(f'Feature columns ({len(FEATURE_COLS)}):', FEATURE_COLS)

## 2. Load & Engineer Features

In [ ]:
def build_daily_series(sales: pd.DataFrame) -> pd.DataFrame:
    """Aggregate to daily per-SKU demand with calendar gap filling."""
    daily = sales.groupby(['product_id', 'sale_date']).agg(
        daily_qty=('quantity', 'sum')
    ).reset_index()
    date_min, date_max = daily['sale_date'].min(), daily['sale_date'].max()
    all_dates = pd.date_range(date_min, date_max, freq='D')
    idx = pd.MultiIndex.from_product(
        [daily['product_id'].unique(), all_dates],
        names=['product_id', 'sale_date']
    )
    full = pd.DataFrame(index=idx).reset_index()
    full = full.merge(daily, on=['product_id', 'sale_date'], how='left')
    full['daily_qty'] = full['daily_qty'].fillna(0).astype(float)
    return full


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(['product_id', 'sale_date']).copy()
    for lag in LAG_FEATURES:
        df[f'lag_{lag}'] = df.groupby('product_id')['daily_qty'].shift(lag)
    for w in ROLLING_WINDOWS:
        df[f'rolling_mean_{w}'] = df.groupby('product_id')['daily_qty'].transform(
            lambda s: s.shift(1).rolling(w, min_periods=1).mean()
        )
        df[f'rolling_std_{w}'] = df.groupby('product_id')['daily_qty'].transform(
            lambda s: s.shift(1).rolling(w, min_periods=1).std()
        ).fillna(0)
    df['day_of_week'] = df['sale_date'].dt.dayofweek
    df['month'] = df['sale_date'].dt.month
    df['quarter'] = df['sale_date'].dt.quarter
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    return df


sales = pd.read_csv(os.path.join(DATA_PROCESSED, 'sales_processed.csv'), parse_dates=['sale_date'])
products = pd.read_csv(os.path.join(DATA_PROCESSED, 'products_processed.csv'))
print(f'Sales:    {sales.shape}')
print(f'Products: {products.shape}')

daily = build_daily_series(sales)
daily = engineer_features(daily)

# Temporal split
cutoff = daily['sale_date'].max() - pd.Timedelta(days=VALIDATION_DAYS)
ready = daily.dropna(subset=FEATURE_COLS)
train = ready[ready['sale_date'] <= cutoff].copy()
val = ready[ready['sale_date'] > cutoff].copy()

X_train, y_train = train[FEATURE_COLS].values, train['daily_qty'].values
X_val, y_val = val[FEATURE_COLS].values, val['daily_qty'].values

print(f'\nTrain: {X_train.shape} | Val: {X_val.shape}')
print(f'Cutoff: {cutoff.date()}')

## 3. Evaluation Helper

In [ ]:
def mape_score(y_true, y_pred, eps=1e-8):
    """MAPE — note: unreliable when actual = 0 (zero-demand days)."""
    return float(np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + eps))) * 100)


def evaluate_model(y_true, y_pred, name):
    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    r2   = r2_score(y_true, y_pred)
    mape = mape_score(y_true, y_pred)
    print(f'{name:<35}  MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}')
    return {'model': name, 'MAE': round(mae,4), 'MSE': round(mse,4),
            'RMSE': round(rmse,4), 'R2': round(r2,4), 'MAPE': round(mape,2)}

## 4. Train All Models

### 4.1 Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = np.clip(lr.predict(X_val), 0, None)
m_lr = evaluate_model(y_val, lr_pred, 'Linear Regression')

### 4.2 Gradient Boosting

In [ ]:
gb = GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.05,
                                subsample=0.8, random_state=RANDOM_STATE)
gb.fit(X_train, y_train)
gb_pred = np.clip(gb.predict(X_val), 0, None)
m_gb = evaluate_model(y_val, gb_pred, 'Gradient Boosting')

### 4.3 Random Forest (Baseline)

In [ ]:
rf_base = RandomForestRegressor(n_estimators=200, max_depth=12,
                                 random_state=RANDOM_STATE, n_jobs=-1)
rf_base.fit(X_train, y_train)
rf_base_pred = np.clip(rf_base.predict(X_val), 0, None)
m_rf_base = evaluate_model(y_val, rf_base_pred, 'Random Forest (Baseline)')

### 4.4 Random Forest — Hyperparameter Tuning (RandomizedSearchCV)

In [ ]:
param_dist = {
    'n_estimators': [100, 150, 200, 300],
    'max_depth': [6, 8, 10, 12, 15, None],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf': [1, 2, 4, 6],
    'max_features': ['sqrt', 'log2', 0.5, 0.7],
}

search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=20,
    scoring='neg_mean_absolute_error',
    cv=3,
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
search.fit(X_train, y_train)

best_params = search.best_params_
print('\nBest parameters:')
for k, v in best_params.items():
    print(f'  {k}: {v}')
print(f'Best CV MAE: {-search.best_score_:.4f}')

rf_tuned = search.best_estimator_
rf_tuned_pred = np.clip(rf_tuned.predict(X_val), 0, None)
m_rf_tuned = evaluate_model(y_val, rf_tuned_pred, 'Random Forest (Tuned)')

### 4.5 XGBoost (Optional)

In [ ]:
m_xgb = None
try:
    from xgboost import XGBRegressor
    xgb = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.05,
                        subsample=0.8, colsample_bytree=0.8,
                        random_state=RANDOM_STATE, verbosity=0)
    xgb.fit(X_train, y_train)
    xgb_pred = np.clip(xgb.predict(X_val), 0, None)
    m_xgb = evaluate_model(y_val, xgb_pred, 'XGBoost')
except ImportError:
    print('XGBoost not installed — skipping.')

## 5. Model Comparison Table

In [ ]:
all_metrics = [m_lr, m_gb, m_rf_base, m_rf_tuned]
if m_xgb:
    all_metrics.append(m_xgb)

df_comparison = pd.DataFrame(all_metrics).set_index('model')
display_cols = ['MAE', 'MSE', 'RMSE', 'R2']
print('\n=== MODEL COMPARISON ===')
print(df_comparison[display_cols].to_string())

champion_row = min(all_metrics, key=lambda x: x['RMSE'])
print(f'\n✓ Champion: {champion_row["model"]}  (RMSE={champion_row["RMSE"]})')

## 6. Hyperparameter Tuning — Before vs After Comparison

In [ ]:
tuning_comparison = pd.DataFrame([m_rf_base, m_rf_tuned]).set_index('model')
print('Hyperparameter Tuning Impact:')
print(tuning_comparison[['MAE', 'RMSE', 'R2']].to_string())

mae_improvement = (m_rf_base['MAE'] - m_rf_tuned['MAE']) / m_rf_base['MAE'] * 100
rmse_improvement = (m_rf_base['RMSE'] - m_rf_tuned['RMSE']) / m_rf_base['RMSE'] * 100
print(f'\nMAE  improvement: {mae_improvement:+.2f}%')
print(f'RMSE improvement: {rmse_improvement:+.2f}%')

## 7. Visualizations

### 7.1 Actual vs Predicted

In [ ]:
champion_pred = rf_tuned_pred  # update based on champion_row if needed

sample_size = min(500, len(y_val))
rng = np.random.default_rng(42)
idx = rng.choice(len(y_val), sample_size, replace=False)

fig, ax = plt.subplots(figsize=(8, 6), facecolor='#1A1A2E')
ax.set_facecolor('#16213E')
ax.scatter(y_val[idx], champion_pred[idx], alpha=0.5, s=18, color='#5C6BC0', label='Predictions')
lim = max(y_val[idx].max(), champion_pred[idx].max()) * 1.05
ax.plot([0, lim], [0, lim], '--', color='#FF7043', lw=1.5, label='Perfect Fit')
ax.set_title('Actual vs Predicted Demand', color='#E8EAF6', fontsize=14, fontweight='bold')
ax.set_xlabel('Actual Demand', color='#E8EAF6')
ax.set_ylabel('Predicted Demand', color='#E8EAF6')
ax.tick_params(colors='#E8EAF6')
ax.legend(facecolor='#16213E', labelcolor='#E8EAF6')
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'actual_vs_predicted.png'), dpi=150, bbox_inches='tight', facecolor='#1A1A2E')
plt.show()
print('Saved: actual_vs_predicted.png')

### 7.2 Residual Plot

In [ ]:
residuals = y_val - champion_pred

fig, ax = plt.subplots(figsize=(8, 6), facecolor='#1A1A2E')
ax.set_facecolor('#16213E')
ax.scatter(champion_pred[idx], residuals[idx], alpha=0.5, s=18, color='#26C6DA')
ax.axhline(0, color='#FF7043', lw=1.5, linestyle='--')
ax.set_title('Residual Plot — Random Forest (Tuned)', color='#E8EAF6', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Demand', color='#E8EAF6')
ax.set_ylabel('Residuals', color='#E8EAF6')
ax.tick_params(colors='#E8EAF6')
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'residual_plot.png'), dpi=150, bbox_inches='tight', facecolor='#1A1A2E')
plt.show()
print('Saved: residual_plot.png')

### 7.3 Error Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6), facecolor='#1A1A2E')
ax.set_facecolor('#16213E')
ax.hist(residuals, bins=50, color='#5C6BC0', edgecolor='#16213E', alpha=0.85)
ax.axvline(0, color='#FF7043', lw=2, linestyle='--', label='Zero Error')
ax.axvline(residuals.mean(), color='#66BB6A', lw=1.5, linestyle='-', label=f'Mean Error: {residuals.mean():.2f}')
ax.set_title('Error Distribution', color='#E8EAF6', fontsize=14, fontweight='bold')
ax.set_xlabel('Prediction Error', color='#E8EAF6')
ax.set_ylabel('Frequency', color='#E8EAF6')
ax.tick_params(colors='#E8EAF6')
ax.legend(facecolor='#16213E', labelcolor='#E8EAF6')
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'error_distribution.png'), dpi=150, bbox_inches='tight', facecolor='#1A1A2E')
plt.show()
print('Saved: error_distribution.png')

### 7.4 Feature Importance

In [ ]:
importances = rf_tuned.feature_importances_
sorted_idx = np.argsort(importances)[::-1]

print('Feature Importance Ranking:')
for rank, i in enumerate(sorted_idx, 1):
    print(f'  {rank:2d}. {FEATURE_COLS[i]:<25} {importances[i]:.6f}')

fig, ax = plt.subplots(figsize=(10, 6), facecolor='#1A1A2E')
ax.set_facecolor('#16213E')
colors = ['#5C6BC0' if i < 5 else '#26C6DA' for i in range(len(FEATURE_COLS))]
ax.barh(
    [FEATURE_COLS[i] for i in sorted_idx[::-1]],
    importances[sorted_idx[::-1]],
    color=colors[::-1],
    edgecolor='#16213E',
)
ax.set_title('Feature Importance — Random Forest (Tuned)', color='#E8EAF6', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score', color='#E8EAF6')
ax.tick_params(colors='#E8EAF6')
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'feature_importance.png'), dpi=150, bbox_inches='tight', facecolor='#1A1A2E')
plt.show()
print('Saved: feature_importance.png')

### 7.5 Model Comparison Chart

In [ ]:
df_plot = pd.DataFrame(all_metrics).set_index('model')
metric_cols = ['MAE', 'RMSE']
colors_palette = ['#5C6BC0', '#26C6DA', '#FF7043', '#66BB6A', '#FFA726']

fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor='#1A1A2E')
for i, metric in enumerate(metric_cols):
    ax = axes[i]
    ax.set_facecolor('#16213E')
    vals = df_plot[metric].values
    bars = ax.bar(range(len(df_plot)), vals, color=colors_palette[:len(vals)], edgecolor='#16213E', width=0.6)
    ax.set_xticks(range(len(df_plot)))
    ax.set_xticklabels(df_plot.index, rotation=30, ha='right', color='#E8EAF6', fontsize=9)
    ax.set_title(metric, color='#E8EAF6', fontsize=13, fontweight='bold')
    ax.tick_params(colors='#E8EAF6')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                f'{val:.3f}', ha='center', va='bottom', color='#E8EAF6', fontsize=9)

fig.suptitle('Model Comparison — Evaluation Metrics', color='#E8EAF6', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'model_comparison.png'), dpi=150, bbox_inches='tight', facecolor='#1A1A2E')
plt.show()
print('Saved: model_comparison.png')

## 8. Save Champion Model

In [ ]:
pkl_path = os.path.join(MODELS_DIR, 'best_forecasting_model.pkl')
joblib.dump(rf_tuned, pkl_path)
print(f'Champion model saved → {pkl_path}')

metadata = {
    'model_name': 'Random Forest (Tuned)',
    'training_date': datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC'),
    'version': '1.0.0',
    'metrics': champion_row,
    'features_used': FEATURE_COLS,
    'best_hyperparameters': best_params,
    'model_file': 'best_forecasting_model.pkl',
    'framework': 'scikit-learn',
}

meta_path = os.path.join(MODELS_DIR, 'model_metadata.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'Metadata saved → {meta_path}')

print('\nModel metadata:')
print(json.dumps(metadata, indent=2, default=str))

## 9. Summary

In [ ]:
print('=' * 65)
print('  DAY 9 MODEL OPTIMIZATION — COMPLETE')
print('=' * 65)
print(f'Champion model  : {champion_row["model"]}')
print(f'MAE             : {champion_row["MAE"]}')
print(f'RMSE            : {champion_row["RMSE"]}')
print(f'R²              : {champion_row["R2"]}')
print()
print('Deliverables generated:')
print('  ✅ models/best_forecasting_model.pkl')
print('  ✅ models/model_metadata.json')
print('  ✅ reports/figures/actual_vs_predicted.png')
print('  ✅ reports/figures/residual_plot.png')
print('  ✅ reports/figures/error_distribution.png')
print('  ✅ reports/figures/feature_importance.png')
print('  ✅ reports/figures/model_comparison.png')
print()
print('Documentation:')
print('  ✅ docs/MODEL_REPORT.md')
print('  ✅ docs/MODEL_EVALUATION.md')
print('  ✅ docs/FEATURE_IMPORTANCE.md')